# RT Notebook 13: Topological Invariants of Continuation Geometry

Status: pre-execution design scaffold. Claim ceiling: `C1_SPECIFICATION_ONLY` until executed with recoverable outputs, hashes, and bounded interpretation.

Research question: can topology alone classify `universal_reconvergence`, `partial_reconvergence`, and `obstruction` across the continuation graphs generated from the bounded Notebook 12 domain?


## 1. Protocol

Test chain: local mechanisms -> continuation graph topology -> reconvergence geometry -> J.

Mechanism labels, mask bits, and active mechanism counts are excluded from topology-only classification features. Failures, ambiguous signatures, and counterexamples are preserved as outputs rather than removed.


In [ ]:
from __future__ import annotations

import json
import math
import zipfile
from collections import Counter, defaultdict, deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Set, Tuple

import networkx as nx
import pandas as pd

SEED = 130013
SPEC_ID = "NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_001"
NB12_ZIP = Path(r"D:/projects/New folder/rt_notebook_12_outputs_results.zip")
OUTPUT_DIR = Path("results") / SPEC_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load Notebook 12 Labels

Notebook 12 output tables provide the bounded configuration domain, the geometry labels, and J. Graphs are regenerated in a later cell because serialized graphs are not present in the archive.


In [ ]:
with zipfile.ZipFile(NB12_ZIP) as zf:
    with zf.open("factorial_results.parquet") as f:
        nb12 = pd.read_parquet(f)
    with zf.open("branch_pair_results.parquet") as f:
        nb12_pairs = pd.read_parquet(f)
    with zf.open("manifest.json") as f:
        nb12_manifest = json.load(f)

print(nb12.shape)
print(nb12[["config_id", "mask_bits", "geometry", "J"]].head())
print(nb12_manifest)


## 3. Notebook 12 Graph Adapter

This adapter must return the exact `networkx.DiGraph` that Notebook 12 would build for a row of `factorial_results.parquet`. The preferred route is to import or paste the Notebook 12 `State`, `MechanismMask`, `SystemConfig`, and `build_continuation_graph` definitions without changing their semantics.


In [ ]:
def build_graph_from_nb12_row(row: Mapping[str, Any]) -> nx.DiGraph:
    """Return the Notebook 12 continuation graph for one factorial row.

    Implementation requirement: bind this function to the Notebook 12 generator
    without adding mechanism labels to the topology-only classifier feature set.
    """
    raise NotImplementedError("Bind to Notebook 12 SystemConfig/build_continuation_graph before execution.")


## 4. Topological Invariant Extraction

The invariant extractor computes topology-only features. Configuration and mechanism labels are retained only for joins, controls, grouping, and audit output.


In [ ]:
def node_depths_from_roots(graph: nx.DiGraph) -> Dict[Any, int]:
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    depths: Dict[Any, int] = {}
    queue = deque((root, 0) for root in roots)
    while queue:
        node, depth = queue.popleft()
        if node in depths and depths[node] <= depth:
            continue
        depths[node] = depth
        for succ in graph.successors(node):
            queue.append((succ, depth + 1))
    return depths

def root_successor_pairs(graph: nx.DiGraph) -> List[Tuple[Any, Any]]:
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    if not roots:
        return []
    root = sorted(roots, key=repr)[0]
    successors = list(graph.successors(root))
    return [(successors[i], successors[j]) for i in range(len(successors)) for j in range(i + 1, len(successors))]

def closure_sets(graph: nx.DiGraph) -> Dict[Any, frozenset]:
    return {node: frozenset({node} | nx.descendants(graph, node)) for node in graph.nodes}

def exact_or_bounded_width(dag: nx.DiGraph, max_antichains: int = 200000) -> Tuple[int, bool]:
    width = 0
    exact = True
    for i, antichain in enumerate(nx.antichains(dag)):
        if i >= max_antichains:
            exact = False
            break
        width = max(width, len(antichain))
    return width, exact

def earliest_common_depth(graph: nx.DiGraph, left: Any, right: Any, depths: Mapping[Any, int]) -> Optional[int]:
    common = ({left} | nx.descendants(graph, left)) & ({right} | nx.descendants(graph, right))
    if not common:
        return None
    return min(depths.get(node, math.inf) for node in common)

def topological_invariants(graph: nx.DiGraph) -> Dict[str, Any]:
    depths = node_depths_from_roots(graph)
    closures = closure_sets(graph)
    unique_closures = set(closures.values())
    condensation = nx.condensation(graph)
    undirected = graph.to_undirected(as_view=False)
    articulation_points = list(nx.articulation_points(undirected)) if undirected.number_of_nodes() else []
    bridges = list(nx.bridges(undirected)) if undirected.number_of_nodes() else []
    width, width_exact = exact_or_bounded_width(condensation)
    pairs = root_successor_pairs(graph)
    merge_depths = [earliest_common_depth(graph, a, b, depths) for a, b in pairs]
    observed_merge_depths = [d for d in merge_depths if d is not None]
    separated_pairs = sum(1 for d in merge_depths if d is None)
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    root = sorted(roots, key=repr)[0] if roots else None
    dominator_count = 0
    dominator_max_depth = 0
    if root is not None:
        try:
            dominators = nx.immediate_dominators(graph, root)
            dominator_count = len(dominators)
            dom_tree = nx.DiGraph((parent, child) for child, parent in dominators.items() if child != parent)
            if dom_tree.number_of_nodes():
                dominator_max_depth = max(nx.shortest_path_length(dom_tree, root, n) for n in dom_tree.nodes if nx.has_path(dom_tree, root, n))
        except Exception:
            dominator_count = -1
            dominator_max_depth = -1
    sink_sccs = [n for n in condensation.nodes if condensation.out_degree(n) == 0]
    source_sccs = [n for n in condensation.nodes if condensation.in_degree(n) == 0]
    return {
        "node_count": graph.number_of_nodes(),
        "edge_count": graph.number_of_edges(),
        "reachability_closure_count": len(unique_closures),
        "reachability_collision_count": graph.number_of_nodes() - len(unique_closures),
        "closure_min_size": min((len(c) for c in unique_closures), default=0),
        "closure_max_size": max((len(c) for c in unique_closures), default=0),
        "scc_count": nx.number_strongly_connected_components(graph),
        "condensation_node_count": condensation.number_of_nodes(),
        "condensation_edge_count": condensation.number_of_edges(),
        "condensation_source_count": len(source_sccs),
        "condensation_sink_count": len(sink_sccs),
        "partial_order_width": width,
        "partial_order_width_exact": width_exact,
        "terminal_basin_count": len(sink_sccs),
        "articulation_point_count": len(articulation_points),
        "articulation_min_depth": min((depths.get(n, math.inf) for n in articulation_points), default=-1),
        "articulation_max_depth": max((depths.get(n, -1) for n in articulation_points), default=-1),
        "bridge_count": len(bridges),
        "root_branch_pair_count": len(pairs),
        "irreversible_separated_pair_count": separated_pairs,
        "has_irreversible_separation": separated_pairs > 0,
        "merge_depth_min": min(observed_merge_depths, default=-1),
        "merge_depth_mean": sum(observed_merge_depths) / len(observed_merge_depths) if observed_merge_depths else -1,
        "merge_depth_max": max(observed_merge_depths, default=-1),
        "dominance_tree_node_count": dominator_count,
        "dominance_tree_max_depth": dominator_max_depth,
        "closure_lattice_node_count": len(unique_closures)
    }


## 5. Exhaustive Regeneration and Join

This cell is the governed execution boundary. Run only after the experiment spec has been frozen and hashed.


In [ ]:
rows = []
skips = []
for _, row in nb12.iterrows():
    try:
        graph = build_graph_from_nb12_row(row)
        inv = topological_invariants(graph)
        inv.update({"config_id": row["config_id"], "geometry": row["geometry"], "J": float(row["J"])})
        rows.append(inv)
    except Exception as exc:
        skips.append({"config_id": row.get("config_id"), "error": repr(exc)})

invariants = pd.DataFrame(rows)
invariants.to_parquet(OUTPUT_DIR / "topological_invariants.parquet", index=False)
(OUTPUT_DIR / "skipped_configurations.json").write_text(json.dumps(skips, indent=2), encoding="utf-8")
print(invariants.shape, "skips", len(skips))


## 6. Topology-Only Classification

Classifier cells may use scikit-learn when available. If unavailable, emit an inconclusive dependency result rather than silently changing the test.


In [ ]:
EXCLUDED = {"config_id", "geometry", "J", "mask_bits", "active_mechanism_count"}
feature_columns = [c for c in invariants.columns if c not in EXCLUDED and pd.api.types.is_numeric_dtype(invariants[c])]

report: Dict[str, Any] = {"spec_id": SPEC_ID, "feature_columns": feature_columns, "claim_ceiling": "C2_CANDIDATE_AFTER_EXECUTION"}
try:
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, balanced_accuracy_score
    from sklearn.model_selection import train_test_split

    X = invariants[feature_columns].fillna(-1)
    y = invariants["geometry"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train, y_train)
    model = RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight="balanced")
    model.fit(X_train, y_train)
    pred_dummy = dummy.predict(X_test)
    pred_model = model.predict(X_test)
    report.update({
        "dummy_balanced_accuracy": balanced_accuracy_score(y_test, pred_dummy),
        "topology_balanced_accuracy": balanced_accuracy_score(y_test, pred_model),
        "classification_report": classification_report(y_test, pred_model, output_dict=True),
        "status": "EXECUTED_CLASSIFIER"
    })
except Exception as exc:
    report.update({"status": "INCONCLUSIVE_DEPENDENCY_OR_EXECUTION_FAILURE", "error": repr(exc)})

(OUTPUT_DIR / "topology_classification_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
report


## 7. Counterexample Catalog

Topology signatures that map to multiple geometry classes or dispersed J values are preserved as limitation evidence.


In [ ]:
signature_columns = [c for c in feature_columns if c not in {"node_count", "edge_count"}]
counterexamples = []
for signature, group in invariants.groupby(signature_columns, dropna=False):
    labels = sorted(group["geometry"].unique())
    if len(labels) > 1 or group["J"].max() != group["J"].min():
        counterexamples.append({
            "signature": dict(zip(signature_columns, signature if isinstance(signature, tuple) else (signature,))),
            "row_count": int(len(group)),
            "geometry_labels": labels,
            "J_min": float(group["J"].min()),
            "J_max": float(group["J"].max()),
            "example_config_ids": group["config_id"].head(10).tolist()
        })

(OUTPUT_DIR / "topology_counterexamples.json").write_text(json.dumps(counterexamples, indent=2), encoding="utf-8")
len(counterexamples)


## 8. Manifest

The manifest binds the run to the immutable spec, source archive, outputs, and claim ceiling.


In [ ]:
manifest = {
    "notebook": "RT Notebook 13",
    "title": "Topological Invariants of Continuation Geometry",
    "spec_id": SPEC_ID,
    "seed": SEED,
    "source_archive": str(NB12_ZIP),
    "claim_ceiling": "C2_BOUNDED_NOTEBOOK_OUTPUT after successful governed execution; C1 before execution",
    "interpretation_constraint": "All conclusions are bounded to the regenerated Notebook 12 continuation-graph domain and do not establish external physical validation."
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest
